# Tuned for 20s Horizon — Summary of Changes (CNN+GRU Experiment)

This version implements a controlled experiment where the recurrent layer is switched to a **GRU**.

1. **Architecture**: Replaced `LSTM` with `GRU(BEST_UNITS, recurrent_dropout=0.1, kernel_regularizer=l2(1e-4))`.
2. **Lookback window**: 40s (20 samples).
3. **Regularization**: Recurrent dropout and L2 weight decay.
4. **Bug fixes**: Corrected checkpoint referencing and ensured evaluation uses the best saved model weights.

# Part 1: Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout)

from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)

import joblib
import random

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [130]:
SEED = 42
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

Configuration

In [133]:
# Approximate Sensor interval
SAMPLE_INTERVAL_SECONDS = 2

# TUNED: shortened from 60s to 40s.
# For a short 20s forecast horizon, a 60s (30-sample) lookback carries a lot of
# older, less-relevant history that mostly adds noise/variance to the LSTM's
# hidden state. A 40s (20-sample) window still captures the recent
# heating/cooling trend (slope) needed to predict 20s ahead, while giving the
# model fewer, more relevant timesteps to learn from -> less overfitting and
# faster training.
LOOKBACK_SECONDS = 40

# Predict 20 seconds into the future
FORECAST_SECONDS = 10

LOOKBACK = LOOKBACK_SECONDS // SAMPLE_INTERVAL_SECONDS
HORIZON = FORECAST_SECONDS // SAMPLE_INTERVAL_SECONDS

print("Lookback samples: ", LOOKBACK)
print("Forecast Horizon samples: ", HORIZON)


Lookback samples:  20
Forecast Horizon samples:  5


# Part 2: Loading Datasets

In [136]:
FILES = {
    "prabhsimrat": "prabh_merged_experiment.csv",
    "manan": "manan_merged_experiment.csv",
    "gursimar":"gursimar_merged_experiment.csv",
    "sushant":"sushant2_merged_experiment.csv",
    "gurvivek":"gurvivek_merged_experiment.csv"
}

In [138]:
datasets ={}

for name, path in FILES.items():
    df = pd.read_csv(path)
    df["timestamp"]= pd.to_datetime(df["timestamp"])
    df["targetLoad"] = df["targetLoad"].fillna(0)
    df = df.sort_values("timestamp").reset_index(drop= True)
    df["source"] = name
    datasets[name]= df
    print(name, "-", df.shape)

prabhsimrat - (14291, 29)
manan - (15275, 29)
gursimar - (14621, 29)
sushant - (14133, 29)
gurvivek - (15391, 29)


In [140]:
FEATURE_COLS = [
    "cpuUsage",
    "cpuPackagePower",
    "gpuCoreTemperature",
    "gpuHotspotTemperature",
    "cpuEfficiencyAverageClock",
    "targetLoad",
    "cpuTemperature"
]

In [142]:
for name, df in datasets.items():

    print(f"\n===== {name} =====")

    print(
        df[
            [
                "cpuUsage",
                "cpuPackagePower",
                "cpuAverageClock",
                "cpuTemperature"
            ]
        ].describe().loc[
            ["min", "mean", "std", "max"]
        ]
    )


===== prabhsimrat =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     0.600000         0.500000      2005.000000       53.300000
mean   48.496858        32.546904      3672.292212       84.164054
std    34.828684        15.110058       351.956097       14.467376
max   100.000000        63.500000      4517.000000       96.300000

===== manan =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     1.000000         7.400000      1096.200000       56.000000
mean   53.342507        37.440537      2687.528492       80.644452
std    32.784520        18.687431       676.848676       14.234102
max   100.000000        66.500000      3225.900000       99.000000

===== gursimar =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     0.500000         4.200000      1300.000000       40.400000
mean   51.469482        30.899815      3474.685521       83.150085
std    34.245191        15.380098       581.144890       15.694

# Part 3: Cleaning

After running (after_stress_generator.ipynb)- most of the cleaning is already done

In [146]:
def clean_run(df):
    df = df.copy()
    # Keep only valid sensor data
    df = df.dropna(subset=FEATURE_COLS)

    # ---- TUNED: additional data cleaning ----
    # Drop physically implausible sensor glitches. A single bad reading (e.g. a
    # negative/zero temperature or >100% usage spike) can inject a large,
    # meaningless dT into the target and encourage the model to chase noise.
    df = df[(df["cpuTemperature"] > 0) & (df["cpuTemperature"] < 110)]
    df = df[(df["cpuUsage"] >= 0) & (df["cpuUsage"] <= 100)]

    # Winsorize (clip, don't drop) the remaining noisy features to their
    # 1st-99th percentile. Clipping instead of dropping preserves the
    # contiguous time index that the sequence builder relies on.
    for col in ["ramUsage", "networkConnections", "processCount", "cpuPackagePower"]:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    df = df.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    return df


In [148]:
for name in datasets:

    datasets[name] = clean_run(
        datasets[name]
    )

    print(
        name,
        len(datasets[name])
    )

prabhsimrat 14291
manan 15275
gursimar 14621
sushant 14133
gurvivek 15391


# Checking Sampling Gaps

In [151]:
def inspect_time_gaps(df, name):

    gaps = (
        df["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    print(f"\n{name}")

    print(
        "Median interval:",
        gaps.median()
    )

    print(
        "Maximum gap:",
        gaps.max()
    )

    print(
        "Gaps > 5 seconds:",
        (gaps > 5).sum()
    )

In [153]:
for name, df in datasets.items():

    inspect_time_gaps(
        df,
        name
    )


prabhsimrat
Median interval: 2.0221316
Maximum gap: 15.3296387
Gaps > 5 seconds: 1

manan
Median interval: 2.029322
Maximum gap: 2.77304
Gaps > 5 seconds: 0

gursimar
Median interval: 2.0232760499999998
Maximum gap: 5.223701
Gaps > 5 seconds: 1

sushant
Median interval: 2.04348185
Maximum gap: 4.4327946
Gaps > 5 seconds: 0

gurvivek
Median interval: 2.0280052499999996
Maximum gap: 287.5861578
Gaps > 5 seconds: 2


# Part 6: Create Balanced Time Blocks

In [157]:
TRAIN_RUNS = ["prabhsimrat","manan","gursimar","sushant","gurvivek" ]
TEST_RUNS = ["prabhsimrat","manan", "gursimar","sushant","gurvivek"]

In [159]:
BLOCK_MINUTES = 10

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

In [161]:
def get_phase_group(mode):

    mode = str(mode).upper()

    if "COOLING" in mode:
        return "COOLING"

    if mode in [
        "INITIAL_IDLE",
        "PRE_EXPERIMENT",
        "POST_EXPERIMENT"
    ]:
        return "IDLE"

    if mode in [
        "RAMP",
        "CHAOS",
        "TRANSITION",
        "MIXED"
    ]:
        return mode

    return "OTHER"

In [163]:
for name, df in datasets.items():

    df = df.copy()

    df["phaseGroup"] = (
        df["mode"]
        .apply(get_phase_group)
    )

    datasets[name] = df

    print(f"\n===== {name} =====")

    print(
        df["phaseGroup"]
        .value_counts()
    )


===== prabhsimrat =====
phaseGroup
MIXED         4374
OTHER         2250
CHAOS         2192
RAMP          2190
TRANSITION    2187
COOLING       1038
IDLE            60
Name: count, dtype: int64

===== manan =====
phaseGroup
MIXED         4365
OTHER         2971
CHAOS         2188
TRANSITION    2185
RAMP          2183
COOLING       1325
IDLE            58
Name: count, dtype: int64

===== gursimar =====
phaseGroup
MIXED         4378
OTHER         2292
CHAOS         2188
TRANSITION    2186
RAMP          2184
COOLING       1334
IDLE            59
Name: count, dtype: int64

===== sushant =====
phaseGroup
MIXED         4316
OTHER         2194
CHAOS         2168
RAMP          2163
TRANSITION    2161
COOLING       1078
IDLE            53
Name: count, dtype: int64

===== gurvivek =====
phaseGroup
MIXED         4365
OTHER         3084
RAMP          2186
CHAOS         2186
TRANSITION    2180
COOLING       1331
IDLE            59
Name: count, dtype: int64


Now Creating Contiguous Blocks

In [166]:
def create_time_blocks(df, run_name, block_minutes = 10):
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop= True)
    all_blocks = []
    block_counter = 0

    # --------------------- Processing Each Contiguous Phase Separately----------------------------

    phase_change = (df["phaseGroup"]!=df["phaseGroup"].shift())

    df["phaseSegment"]= (phase_change.cumsum())

    for segment_id, segment in df.groupby("phaseSegment"):
        segment = (segment.sort_values("timestamp").copy())
        phase = (segment["phaseGroup"].iloc[0])
        segment_start = (segment["timestamp"].min())

        # Assign 10 minute block number
        elapsed_minutes = (segment["timestamp"] - segment_start).dt.total_seconds() / 60

        segment["localBlock"] = (elapsed_minutes // block_minutes).astype(int)

        for local_block, block in segment.groupby("localBlock"):
            block = block.copy()
            block["blockId"] = (f"{run_name}"
                                f"{block_counter}")
            block["runName"] = run_name
            all_blocks.append(block)
            block_counter+=1

    return all_blocks

In [168]:
all_blocks = []

for name in TRAIN_RUNS:

    run_blocks = create_time_blocks(

        df=datasets[name],

        run_name=name,

        block_minutes=BLOCK_MINUTES
    )

    all_blocks.extend(
        run_blocks
    )

    print(
        name,
        "blocks:",
        len(run_blocks)
    )

prabhsimrat blocks: 57
manan blocks: 62
gursimar blocks: 59
sushant blocks: 58
gurvivek blocks: 61


# 7. Remove Blocks that are too short

In [171]:
MIN_REQUIRED_SAMPLES = (
    LOOKBACK
    + HORIZON
    + 5
)

In [173]:
valid_blocks = []

removed_blocks = []

for block in all_blocks:

    if len(block) >= MIN_REQUIRED_SAMPLES:

        valid_blocks.append(block)

    else:

        removed_blocks.append(block)


print(
    "Valid blocks:",
    len(valid_blocks)
)

print(
    "Removed short blocks:",
    len(removed_blocks)
)

Valid blocks: 293
Removed short blocks: 4


In [175]:
block_summary = []

for block in valid_blocks:

    block_summary.append({

        "blockId":
            block["blockId"].iloc[0],

        "runName":
            block["runName"].iloc[0],

        "phaseGroup":
            block["phaseGroup"].iloc[0],

        "samples":
            len(block),

        "startTime":
            block["timestamp"].min(),

        "endTime":
            block["timestamp"].max()
    })


block_summary = pd.DataFrame(
    block_summary
)

display(
    block_summary.head(20)
)

,blockId,runName,phaseGroup,samples,startTime,endTime
0,prabhsimrat1,prabhsimrat,IDLE,60,2026-07-13 01:46:21.516468900+05:30,2026-07-13 01:48:20.954834300+05:30
1,prabhsimrat2,prabhsimrat,RAMP,297,2026-07-13 01:48:22.976551300+05:30,2026-07-13 01:58:21.543317200+05:30
2,prabhsimrat3,prabhsimrat,RAMP,294,2026-07-13 01:58:23.563991+05:30,2026-07-13 02:08:21.667127800+05:30
3,prabhsimrat4,prabhsimrat,RAMP,297,2026-07-13 02:08:23.693350100+05:30,2026-07-13 02:18:22.170001900+05:30
4,prabhsimrat5,prabhsimrat,RAMP,297,2026-07-13 02:18:24.190197800+05:30,2026-07-13 02:28:22.354821300+05:30
5,prabhsimrat6,prabhsimrat,RAMP,294,2026-07-13 02:28:24.373413100+05:30,2026-07-13 02:38:21.209319500+05:30
6,prabhsimrat7,prabhsimrat,RAMP,297,2026-07-13 02:38:23.234455900+05:30,2026-07-13 02:48:22.411768600+05:30
7,prabhsimrat8,prabhsimrat,RAMP,297,2026-07-13 02:48:24.430841100+05:30,2026-07-13 02:58:22.673117500+05:30
8,prabhsimrat9,prabhsimrat,RAMP,117,2026-07-13 02:58:24.693306400+05:30,2026-07-13 03:02:19.217204900+05:30
9,prabhsimrat10,prabhsimrat,COOLING,149,2026-07-13 03:02:21.235932100+05:30,2026-07-13 03:07:20.273973200+05:30


In [177]:
print(
    block_summary[
        "phaseGroup"
    ].value_counts()
)

phaseGroup
MIXED         75
OTHER         54
RAMP          40
CHAOS         40
TRANSITION    40
COOLING       39
IDLE           5
Name: count, dtype: int64


# 8. Split whole blocks into train, validation and internal test

In [180]:
from sklearn.model_selection import train_test_split

In [182]:
def split_blocks_balanced(
    block_summary,
    seed=42
):

    train_ids = []
    val_ids = []
    test_ids = []

    # ----------------------------------------
    # Split separately by:
    # device + experiment phase
    # ----------------------------------------

    grouped = block_summary.groupby(
        [
            "runName",
            "phaseGroup"
        ]
    )


    for (
        run_name,
        phase
    ), group in grouped:

        ids = (
            group["blockId"]
            .tolist()
        )


        # Reproducible random order
        rng = np.random.default_rng(
            seed
        )

        rng.shuffle(ids)


        n = len(ids)


        print(
            run_name,
            phase,
            "blocks:",
            n
        )


        # ------------------------------------
        # Very small groups
        # ------------------------------------

        if n == 1:

            train_ids.extend(ids)

            continue


        if n == 2:

            train_ids.append(ids[0])
            val_ids.append(ids[1])

            continue


        # ------------------------------------
        # Normal groups
        # ------------------------------------

        n_train = max(
            1,
            int(round(n * TRAIN_RATIO))
        )

        n_val = max(
            1,
            int(round(n * VAL_RATIO))
        )


        # Ensure at least one test block
        if (
            n_train
            + n_val
            >= n
        ):

            n_train = n - 2
            n_val = 1


        train_ids.extend(
            ids[:n_train]
        )


        val_ids.extend(
            ids[
                n_train:
                n_train + n_val
            ]
        )


        test_ids.extend(
            ids[
                n_train + n_val:
            ]
        )


    return (
        train_ids,
        val_ids,
        test_ids
    )

In [184]:
(
    train_block_ids,
    val_block_ids,
    test_block_ids

) = split_blocks_balanced(
    block_summary,
    seed=SEED
)

gursimar CHAOS blocks: 8
gursimar COOLING blocks: 8
gursimar IDLE blocks: 1
gursimar MIXED blocks: 15
gursimar OTHER blocks: 11
gursimar RAMP blocks: 8
gursimar TRANSITION blocks: 8
gurvivek CHAOS blocks: 8
gurvivek COOLING blocks: 8
gurvivek IDLE blocks: 1
gurvivek MIXED blocks: 15
gurvivek OTHER blocks: 13
gurvivek RAMP blocks: 8
gurvivek TRANSITION blocks: 8
manan CHAOS blocks: 8
manan COOLING blocks: 8
manan IDLE blocks: 1
manan MIXED blocks: 15
manan OTHER blocks: 12
manan RAMP blocks: 8
manan TRANSITION blocks: 8
prabhsimrat CHAOS blocks: 8
prabhsimrat COOLING blocks: 7
prabhsimrat IDLE blocks: 1
prabhsimrat MIXED blocks: 15
prabhsimrat OTHER blocks: 9
prabhsimrat RAMP blocks: 8
prabhsimrat TRANSITION blocks: 8
sushant CHAOS blocks: 8
sushant COOLING blocks: 8
sushant IDLE blocks: 1
sushant MIXED blocks: 15
sushant OTHER blocks: 9
sushant RAMP blocks: 8
sushant TRANSITION blocks: 8


In [186]:
print(
    "\nTraining blocks:",
    len(train_block_ids)
)

print(
    "Validation blocks:",
    len(val_block_ids)
)

print(
    "Internal test blocks:",
    len(test_block_ids)
)


Training blocks: 211
Validation blocks: 38
Internal test blocks: 44


In [188]:
block_summary["split"] = (
    "UNASSIGNED"
)

block_summary.loc[

    block_summary["blockId"]
    .isin(train_block_ids),

    "split"

] = "TRAIN"


block_summary.loc[

    block_summary["blockId"]
    .isin(val_block_ids),

    "split"

] = "VALIDATION"


block_summary.loc[

    block_summary["blockId"]
    .isin(test_block_ids),

    "split"

] = "TEST"

In [190]:
coverage = pd.crosstab(

    block_summary["phaseGroup"],

    block_summary["split"]
)

display(coverage)

split,TEST,TRAIN,VALIDATION
phaseGroup,,,
CHAOS,5,30,5
COOLING,5,29,5
IDLE,0,5,0
MIXED,15,50,10
OTHER,9,37,8
RAMP,5,30,5
TRANSITION,5,30,5


In [192]:
device_coverage = pd.crosstab(

    block_summary["runName"],

    block_summary["split"]
)

display(device_coverage)

split,TEST,TRAIN,VALIDATION
runName,,,
gursimar,8,43,8
gurvivek,9,44,8
manan,9,43,8
prabhsimrat,9,40,7
sushant,9,41,7


# 10. Converting IDs back to block lists

In [195]:
train_blocks = []

val_blocks = []

test_blocks = []


for block in valid_blocks:

    block_id = (
        block["blockId"]
        .iloc[0]
    )


    if block_id in train_block_ids:

        train_blocks.append(
            block
        )


    elif block_id in val_block_ids:

        val_blocks.append(
            block
        )


    elif block_id in test_block_ids:

        test_blocks.append(
            block
        )

In [197]:
print(
    "Train blocks:",
    len(train_blocks)
)

print(
    "Validation blocks:",
    len(val_blocks)
)

print(
    "Internal test blocks:",
    len(test_blocks)
)

Train blocks: 211
Validation blocks: 38
Internal test blocks: 44


# Part 11: Fitting the scaler on traininig blocks only

In [200]:
scaler_training_data = pd.concat(

    [
        block[FEATURE_COLS]
        for block in train_blocks
    ],

    ignore_index=True
)

In [202]:
scaler_X = StandardScaler()

scaler_X.fit(
    scaler_training_data
)

StandardScaler()

In [204]:
joblib.dump(

    scaler_X,

    "cross_device_feature_scaler.pkl"
)

print(
    "Scaler fitted only on training blocks."
)

Scaler fitted only on training blocks.


# Part 12: Create sequences inside each block

In [207]:
def create_sequences_from_block(
    block,
    scaler,
    lookback,
    horizon,
    max_gap_seconds=5
):

    block = (
        block
        .sort_values("timestamp")
        .reset_index(drop=True)
        .copy()
    )


    scaled_features = scaler.transform(
        block[FEATURE_COLS]
    )


    temperatures = (

        block["cpuTemperature"]

        .to_numpy()
    )


    timestamps = (

        block["timestamp"]

        .to_numpy()
    )


    X = []
    y = []

    current_temperatures = []

    future_temperatures = []

    prediction_timestamps = []


    for i in range(

        lookback,

        len(block) - horizon + 1
    ):


        sequence_start = (
            i - lookback
        )


        current_index = (
            i - 1
        )


        future_index = (

            current_index

            + horizon
        )


        if future_index >= len(block):

            break


        # ------------------------------------
        # Check entire history → future period
        # ------------------------------------

        relevant_times = (

            block["timestamp"]

            .iloc[
                sequence_start:
                future_index + 1
            ]
        )


        gaps = (

            relevant_times

            .diff()

            .dt.total_seconds()

            .dropna()
        )


        if (
            gaps > max_gap_seconds
        ).any():

            continue


        # ------------------------------------
        # Input
        # ------------------------------------

        X.append(

            scaled_features[
                sequence_start:i
            ]
        )


        # ------------------------------------
        # Target
        # ------------------------------------

        current_temp = (

            temperatures[
                current_index
            ]
        )


        future_temp = (

            temperatures[
                future_index
            ]
        )


        delta_temp = (

            future_temp

            - current_temp
        )


        y.append(
            delta_temp
        )


        current_temperatures.append(
            current_temp
        )


        future_temperatures.append(
            future_temp
        )


        prediction_timestamps.append(
            timestamps[
                future_index
            ]
        )


    return (

        np.asarray(X),

        np.asarray(y),

        np.asarray(
            current_temperatures
        ),

        np.asarray(
            future_temperatures
        ),

        np.asarray(
            prediction_timestamps
        )
    )

# 13. Create datasets from block lists

In [210]:
def create_dataset_from_blocks(
    blocks,
    scaler
):

    all_X = []
    all_y = []

    all_current = []
    all_future = []

    all_timestamps = []

    all_phases = []
    all_runs = []


    for block in blocks:


        (
            X,
            y,
            current,
            future,
            timestamps

        ) = create_sequences_from_block(

            block=block,

            scaler=scaler,

            lookback=LOOKBACK,

            horizon=HORIZON
        )


        if len(X) == 0:

            continue


        phase = (
            block[
                "phaseGroup"
            ].iloc[0]
        )


        run_name = (
            block[
                "runName"
            ].iloc[0]
        )


        all_X.append(X)

        all_y.append(y)

        all_current.append(
            current
        )

        all_future.append(
            future
        )

        all_timestamps.append(
            timestamps
        )


        all_phases.extend(

            [phase] * len(X)
        )


        all_runs.extend(

            [run_name] * len(X)
        )


    return (

        np.concatenate(all_X),

        np.concatenate(all_y),

        np.concatenate(all_current),

        np.concatenate(all_future),

        np.concatenate(all_timestamps),

        np.asarray(all_phases),

        np.asarray(all_runs)
    )

In [212]:
(
    X_train,
    y_train,
    train_current,
    train_actual,
    train_timestamps,
    train_phases,
    train_runs

) = create_dataset_from_blocks(

    train_blocks,

    scaler_X
)

In [214]:
# TUNED: data cleaning on the training target only.
# Extreme dT values are almost always transient sensor glitches rather than
# real 20s temperature swings. We winsorize the TRAINING target to the
# 1st-99th percentile so the model isn't pulled toward noise, while leaving
# validation/test targets untouched so evaluation metrics stay honest.
y_train_lower = np.percentile(y_train, 1)
y_train_upper = np.percentile(y_train, 99)

print(f"Clipping training dT targets to [{y_train_lower:.3f}, {y_train_upper:.3f}] deg C (1st-99th pct)")

y_train = np.clip(y_train, y_train_lower, y_train_upper)


Clipping training dT targets to [-19.000, 19.500] deg C (1st-99th pct)


In [216]:
(
    X_val,
    y_val,
    val_current,
    val_actual,
    val_timestamps,
    val_phases,
    val_runs

) = create_dataset_from_blocks(

    val_blocks,

    scaler_X
)

In [218]:
(
    X_test,
    y_test,
    test_current,
    test_actual,
    test_timestamps,
    test_phases,
    test_runs

) = create_dataset_from_blocks(

    test_blocks,

    scaler_X
)

In [220]:
print("\nTRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nVALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nINTERNAL TEST")
print("X:", X_test.shape)
print("y:", y_test.shape)


TRAIN
X: (46626, 20, 7)
y: (46626,)

VALIDATION
X: (9251, 20, 7)
y: (9251,)

INTERNAL TEST
X: (10726, 20, 7)
y: (10726,)


# 14. Verify Actual Sequence Diversity

In [223]:
def show_sequence_distribution(
    phases,
    name
):

    print(
        f"\n===== {name} ====="
    )

    print(

        pd.Series(phases)

        .value_counts()
    )

In [225]:
show_sequence_distribution(
    train_phases,
    "TRAIN"
)

show_sequence_distribution(
    val_phases,
    "VALIDATION"
)

show_sequence_distribution(
    test_phases,
    "INTERNAL TEST"
)


===== TRAIN =====
MIXED         13246
OTHER          7576
CHAOS          7251
TRANSITION     7235
RAMP           7218
COOLING        3931
IDLE            169
Name: count, dtype: int64

===== VALIDATION =====
MIXED         2697
OTHER         1870
CHAOS         1358
RAMP          1357
TRANSITION    1349
COOLING        620
Name: count, dtype: int64

===== INTERNAL TEST =====
MIXED         4033
OTHER         2008
RAMP          1358
TRANSITION    1355
CHAOS         1353
COOLING        619
Name: count, dtype: int64


# Part 15: Building the CNN+GRU Model

In [228]:
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, GRU, Dense, Dropout)
from tensorflow.keras.models import Model

# Using the parameters from the previous tuned run
BEST_UNITS = 40
# The previous tuned run used 0.30 dropout
BEST_DROPOUT = 0.30

inputs = Input(shape=(LOOKBACK, len(FEATURE_COLS)))

# 1D Convolution to extract local patterns
x = Conv1D(
    filters=32,
    kernel_size=3,
    activation='relu',
    padding='same'
)(inputs)

x = MaxPooling1D(pool_size=2)(x)

# Controlled Experiment: Replacing LSTM with GRU
x = GRU(
    BEST_UNITS,
    recurrent_dropout=0.1,
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(x)

x = Dropout(BEST_DROPOUT)(x)

x = Dense(
    16,
    activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(1e-4)
)(x)

x = Dropout(0.2)(x)

outputs = Dense(1)(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 20, 32)         │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 40)             │         8,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,257 (40.07 KB)

 Trainable params: 10,257 (40.07 KB)

 Non-trainable params: 0 (0.00 B)

In [230]:
# TUNED: learning rate nudged up slightly (0.0005 -> 0.0008) since the
# shorter lookback + added regularization mean the model needs a bit more
# step size to converge in the same number of epochs. clipnorm=1.0 added for
# training stability (guards against occasional large gradients from any
# remaining outliers).
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0008,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.Huber(delta=2.0),
    metrics=["mae"]
)


# Part 16: Train

In [233]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    min_delta=0.01,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# Updated filename for the GRU model checkpoint
checkpoint = ModelCheckpoint(
    'best_cross_device_gru.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)


Train

In [236]:
# TUNED: batch_size 64 -> 32. Smaller batches give noisier, more frequent
# gradient updates, which for this dataset size tends to generalize slightly
# better than large batches (a mild regularizing effect) at some cost to
# training speed.
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=32,

    shuffle=True,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)


Epoch 1/100
1449/1458 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.8981 - mae: 2.5566
Epoch 1: val_loss improved from inf to 3.10708, saving model to best_cross_device_gru.keras
1458/1458 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 3.8976 - mae: 2.5564 - val_loss: 3.1071 - val_mae: 2.1309 - learning_rate: 8.0000e-04
Epoch 2/100
1449/1458 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.3609 - mae: 2.2941
Epoch 2: val_loss improved from 3.10708 to 2.92188, saving model to best_cross_device_gru.keras
1458/1458 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 3.3605 - mae: 2.2938 - val_loss: 2.9219 - val_mae: 2.0256 - learning_rate: 8.0000e-04
Epoch 3/100
1449/1458 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.1415 - mae: 2.1710
Epoch 3: val_loss improved from 2.92188 to 2.87113, saving model to best_cross_device_gru.keras
1458/1458 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 3.1416 - mae: 2.1710 - val_loss: 2.8711 - val_mae: 2.0035 - learning_rate: 8.0000e-04
Epoch 4/100
1443/1458 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/ste

In [238]:
def calculate_metrics(actual, predicted, name):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    print(f"\n===== {name} =====")
    print(f"MAE:  {mae:.3f} °C")
    print(f"RMSE: {rmse:.3f} °C")
    print(f"R²:   {r2:.4f}")

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

**FIX:** The next cell loads the checkpointed *best* model and computes `test_gru_predictions` correctly. The evaluation below reflects the best GRU checkpoint.

In [241]:
# Load the actual best checkpoint explicitly (GRU version)
best_model = tf.keras.models.load_model(
    'best_cross_device_gru.keras'
)

# Predict temperature change
test_predicted_delta = (
    best_model.predict(X_test)
    .flatten()
)

# Reconstruct future temperature
test_gru_predictions = (
    test_current
    + test_predicted_delta
)

# Persistence baseline:
test_persistence = test_current.copy()

336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [243]:
internal_baseline = calculate_metrics(

    test_actual,

    test_persistence,

    "Internal Test - Persistence"
)


===== Internal Test - Persistence =====
MAE:  2.795 °C
RMSE: 5.648 °C
R²:   0.8512


In [245]:
internal_gru = calculate_metrics(
    test_actual,
    test_gru_predictions,
    'Internal Test - GRU'
)


===== Internal Test - GRU =====
MAE:  2.203 °C
RMSE: 4.809 °C
R²:   0.8921


In [258]:
import numpy as np
import tensorflow as tf
import tf2onnx
import onnxruntime as ort


KERAS_MODEL_PATH = "best_cross_device_lstm.keras"
ONNX_MODEL_PATH = "best_cross_device_grucnn_all5.onnx"


# ---------------------------------------------------------
# 1. LOAD KERAS MODEL
# ---------------------------------------------------------

print("\nLoading Keras model...")

keras_model = tf.keras.models.load_model(
    KERAS_MODEL_PATH,
    compile=False
)

print("Keras model loaded successfully.")
print("Input shape:", keras_model.input_shape)
print("Output shape:", keras_model.output_shape)


# ---------------------------------------------------------
# 2. CREATE EXPLICIT INFERENCE FUNCTION
# ---------------------------------------------------------

input_signature = [
    tf.TensorSpec(
        shape=(None, 20, 6),
        dtype=tf.float32,
        name="telemetry_input"
    )
]


@tf.function(
    input_signature=input_signature
)
def inference_function(telemetry_input):

    prediction = keras_model(
        telemetry_input,
        training=False
    )

    return {
        "predicted_delta_t": prediction
    }


# ---------------------------------------------------------
# 3. CONVERT FUNCTION TO ONNX
# ---------------------------------------------------------

print("\nConverting model to ONNX...")

tf2onnx.convert.from_function(
    inference_function,
    input_signature=input_signature,
    opset=17,
    output_path=ONNX_MODEL_PATH
)

print("ONNX model saved successfully.")
print("Saved at:", ONNX_MODEL_PATH)


# ---------------------------------------------------------
# 4. CREATE IDENTICAL TEST INPUT
# ---------------------------------------------------------

np.random.seed(42)

test_input = np.random.randn(
    1,
    20,
    6
).astype(np.float32)

print("\nTest input shape:", test_input.shape)


# ---------------------------------------------------------
# 5. KERAS PREDICTION
# ---------------------------------------------------------

keras_prediction = keras_model.predict(
    test_input,
    verbose=0
)

keras_delta_t = float(
    keras_prediction[0][0]
)


# ---------------------------------------------------------
# 6. ONNX PREDICTION
# ---------------------------------------------------------

onnx_session = ort.InferenceSession(
    ONNX_MODEL_PATH,
    providers=["CPUExecutionProvider"]
)

onnx_input = onnx_session.get_inputs()[0]
onnx_output = onnx_session.get_outputs()[0]

print("\nONNX input:")
print("Name :", onnx_input.name)
print("Shape:", onnx_input.shape)
print("Type :", onnx_input.type)

print("\nONNX output:")
print("Name :", onnx_output.name)
print("Shape:", onnx_output.shape)
print("Type :", onnx_output.type)


onnx_prediction = onnx_session.run(
    None,
    {
        onnx_input.name: test_input
    }
)


onnx_delta_t = float(
    np.asarray(onnx_prediction[0]).reshape(-1)[0]
)


# ---------------------------------------------------------
# 7. COMPARE
# ---------------------------------------------------------

difference = abs(
    keras_delta_t - onnx_delta_t
)

print("\n========================================")
print("       CONVERSION VALIDATION RESULT")
print("========================================")

print(f"Keras Delta T : {keras_delta_t:.10f}")
print(f"ONNX Delta T  : {onnx_delta_t:.10f}")
print(f"Difference    : {difference:.10f}")


if difference < 1e-4:

    print(
        "\nSUCCESS: ONNX model matches Keras model."
    )

else:

    print(
        "\nWARNING: Predictions differ more than expected."
    )


Loading Keras model...
Keras model loaded successfully.
Input shape: (None, 20, 6)
Output shape: (None, 1)

Converting model to ONNX...


TF freezing failed. Attempting to fix freezing errors.
Removed Fill functional_1_1/lstm_1_1/AssignVariableOp_3
Removed Shape functional_1_1/lstm_1_1/AssignVariableOp_2
Removed ExpandDims functional_1_1/lstm_1_1/AssignVariableOp_1
Removed Squeeze functional_1_1/lstm_1_1/AssignVariableOp


ONNX model saved successfully.
Saved at: best_cross_device_grucnn_all5.onnx

Test input shape: (1, 20, 6)

ONNX input:
Name : telemetry_input
Shape: ['unk__880', 20, 6]
Type : tensor(float)

ONNX output:
Name : predicted_delta_t
Shape: ['unk__881', 1]
Type : tensor(float)

       CONVERSION VALIDATION RESULT
Keras Delta T : -6.3306937218
ONNX Delta T  : -6.3306937218
Difference    : 0.0000000000

SUCCESS: ONNX model matches Keras model.


In [248]:
def evaluate_by_phase(

    actual,
    predicted,
    phases,
    model_name
):

    results = []


    for phase in np.unique(
        phases
    ):

        mask = (
            phases == phase
        )


        if mask.sum() < 10:

            continue


        mae = mean_absolute_error(

            actual[mask],

            predicted[mask]
        )


        rmse = np.sqrt(

            mean_squared_error(

                actual[mask],

                predicted[mask]
            )
        )


        results.append({

            "Model":
                model_name,

            "Phase":
                phase,

            "Samples":
                mask.sum(),

            "MAE":
                mae,

            "RMSE":
                rmse
        })


    return pd.DataFrame(
        results
    )

In [249]:
gru_phase_results = evaluate_by_phase(
    test_actual,
    test_gru_predictions,
    test_phases,
    "GRU"
)
display(gru_phase_results)

,Model,Phase,Samples,MAE,RMSE
0,GRU,CHAOS,1353,2.426680,5.263991
1,GRU,COOLING,619,1.722202,3.248201
2,GRU,MIXED,4033,2.260416,4.979320
3,GRU,OTHER,2008,2.005404,4.259939
4,GRU,RAMP,1358,2.130403,4.390785
5,GRU,TRANSITION,1355,2.395196,5.526925


In [250]:
baseline_phase_results = evaluate_by_phase(

    test_actual,

    test_persistence,

    test_phases,

    "Persistence"
)

In [255]:
phase_comparison = pd.concat(
    [
        baseline_phase_results,
        gru_phase_results
    ],
    ignore_index=True
)

display(
    phase_comparison.sort_values(["Phase", "Model"])
)

,Model,Phase,Samples,MAE,RMSE
6,GRU,CHAOS,1353,2.426680,5.263991
0,Persistence,CHAOS,1353,2.989505,5.943661
7,GRU,COOLING,619,1.722202,3.248201
1,Persistence,COOLING,619,2.643942,4.637945
8,GRU,MIXED,4033,2.260416,4.979320
2,Persistence,MIXED,4033,3.039425,5.998609
9,GRU,OTHER,2008,2.005404,4.259939
3,Persistence,OTHER,2008,2.333466,4.978666
10,GRU,RAMP,1358,2.130403,4.390785
4,Persistence,RAMP,1358,2.447496,4.800030
